In this notebook we are going to see what is there inside the free text notes folder. The aim is to understand each variable of the different tables, discover the data inside, and maybe detect anomalies inside.

What we can do is making a dataset that contains useful information for each csv file

In [ ]:
import pandas as pd

We create a function that gives data for a dataframe and the description of each variables given in parameters.

In [ ]:
def analyze_dataframe(df, description_list):
    analysis = pd.DataFrame({
        "column_name": df.columns,
        "column_type": df.dtypes.values,
        "description": description_list,
        "number_of_unique_values": df.nunique().values,
        "number_of_missing_values": df.isna().sum().values,
        "percentage_missing_values": (df.isna().mean() * 100).values,
        "unique_values_list": [df[col].unique() for col in df.columns],
        "percentage_unique_values": (df.nunique() / len(df) * 100).values
    })
    
    return analysis

So now we can take all the csv files

In [ ]:
base_path = "../../data/mimic-iv-note-deidentified-free-text-clinical-notes-2.2/note/"
files = [
    "discharge.csv.gz",
    "discharge_detail.csv.gz",
    "radiology.csv.gz",
    "radiology_detail.csv.gz"
]

notes_path = [base_path + f for f in files]

Write the description for each variables of each csv files

In [ ]:
variables_description = {}

variables_description[base_path + "discharge.csv.gz"] = [
    "Unique note identifier",
    "Patient ID",
    "Hospital admission ID",
    "Note type",
    "Sequence number within admission",
    "Event timestamp (clinical time)",
    "Record timestamp (system time)",
    "Clinical note content"
]

variables_description[base_path + "discharge_detail.csv.gz"] = [
    "Unique note identifier",
    "Patient ID",
    "Field name",
    "Field value",
    "Field order"
]

variables_description[base_path + "radiology.csv.gz"] = [
    "Unique note identifier",
    "Patient ID",
    "Hospital admission ID",
    "Note type",
    "Sequence number",
    "Event timestamp",
    "Record timestamp",
    "Radiology report text"
]

variables_description[base_path + "radiology_detail.csv.gz"] = [
    "Unique note identifier",
    "Patient ID",
    "Field name (exam_code, exam_name, etc.)",
    "Field value",
    "Field order"
]

And compute all we need for the summary

In [ ]:
for note in notes_path:
    df = pd.read_csv(note)
    print("\n\n##########################################################")
    print(f"\n\n{note}\n\n{df.shape[0]} rows\n\n")
    df_analysis = analyze_dataframe(df,variables_description[note])
    print(df_analysis)

print("\n\n\n\n\n\n\n")

This exploratory analysis provided a better understanding of the structure and content of the clinical notes datasets. The data is organized into two main types of tables:

* Main tables (`discharge.csv`, `radiology.csv`) containing the full clinical notes
* Detail tables (`*_detail.csv`) storing additional information using a key-value format

Some relevant points can be highlighted about the dataset:

**\* Missing Values in `radiology.csv`**

The `hadm_id` variable contains a high percentage of missing values (~50%), which may complicate linking notes to hospital admissions.

**\* Low Variability Variables**

Some variables provide very little information:

* `note_type` is constant in `discharge.csv` and has only two values in `radiology.csv`
* `field_name`, `field_value`, and `field_ordinal` in `discharge_detail.csv` contain only one unique value

These variables have limited analytical value.

**\* Detail Tables Structure**

The detail tables use a key-value format:

* `field_name` → type of information (e.g., exam_code)
* `field_value` → actual value

However, in `discharge_detail.csv`, this structure does not add meaningful information and appears redundant.  

Now that the analysis is complete, we can proceed to feature selection:

**Feature Selection**

**\* Variables to Keep**

The following variables are essential:

* `note_id` → unique identifier
* `subject_id` → patient identifier 
* `hadm_id` → hospital admission (despite missing values)
* `charttime` → clinical timestamp
* `text` → clinical note content (key for NLP)

**\* Variables to Use with Caution**

* `storetime` → system timestamp (less relevant than `charttime`)
* `note_seq` → may be useful for ordering notes (we will may use it after)

**\* Variables to Remove**

The following can be excluded:

* `note_type` (low variability)
* `discharge_detail.csv` (no useful information)
* `field_name`, `field_value`, `field_ordinal` (not useful in current form)

**\* Target Dataset Structure**

The final goal is to build a unified dataset with the following structure:

| note_id | subject_id | hadm_id | charttime | text | source |

Where `source` indicates whether the note comes from **discharge** or **radiology**.

With respect to that, we can now create our dataset

In [ ]:
# Load datasets
df_discharge = pd.read_csv(base_path + "discharge.csv.gz")
df_radiology = pd.read_csv(base_path + "radiology.csv.gz")

# Add source column
df_discharge["source"] = "discharge"
df_radiology["source"] = "radiology"

# Select relevant columns
cols = ["note_id", "subject_id", "hadm_id", "charttime", "text", "source"]

df_discharge = df_discharge[cols]
df_radiology = df_radiology[cols]

# Concatenate datasets
df_final = pd.concat([df_discharge, df_radiology], ignore_index=True)

# Show the dataset
print(df_final.head())

print(f"\n\n{df_final.shape[0]} rows and {df_final.shape[1]} columns")

# Final analysis
analysis_final = analyze_dataframe(df_final,
    ["Unique note identifier",
     "Patient ID",
     "Hospital admission ID",
     "Event timestamp",
     "Clinical note content",
     "Source of note"])

print(f"\n\n{analysis_final}")

# Save the final dataset to a CSV file
output_path = "../../data/free_text_clinical_notes.csv.gz"
df_final.to_csv(output_path, index=False, compression="gzip")